# Wind curtailment dataset (Elexon)

Builds one table: every wind farm bid-offer acceptance (June 2022 – June 2025) with its curtailed volume and the Elexon features, saved as `elexon_data.csv`.

**General idea (for every Elexon pull):**
- determine base url and endpoint for data you want
- determine required parameters and other parameters you want to filter by
- then pull the data, `response.json()` returns a list of dicts (or a dict with a `data` key)
- save as dataframe, each dict is a row, the keys are columns

**Documentation:** https://developer.data.elexon.co.uk, https://bmrs.elexon.co.uk/api-documentation

| Dataset | Endpoint | Use | Request limit found |
|---|---|---|---|
| Bid-offer acceptances (BOALF) | `/datasets/BOALF/stream` | Accepted levels, SO flag | none (one request) |
| BM units | `/reference/bmunits/all` | Wind farm IDs (`fuelType == 'WIND'`) | none |
| Physical notifications (PN) | `/datasets/PN/stream` | Planned output | 3 years returns empty, so yearly |
| Metered generation (B1610) | `/datasets/B1610/stream` | Actual output per unit | 1 year fails, so monthly |
| Wind generation forecast (WINDFOR) | `/datasets/WINDFOR/stream` | Day-ahead wind forecast (GB total) | none (one request) |
| TSDF / NDF | `/datasets/TSDF/stream`, `/datasets/NDF/stream` | Day-ahead demand forecasts | 3 years returns empty, so quarterly |
| INDO / ITSDO | `/datasets/INDO`, `/datasets/ITSDO` | Actual demand (lagged) | 28 days |
| Interconnectors | `/generation/outturn/interconnectors` | Interconnector flows (lagged) | 7 days |

## Setup

In [1]:
import os
import pandas as pd
import requests

file_path =  # folder for the raw and output csvs
base_url = "https://data.elexon.co.uk/bmrs/api/v1"

START, END = "2022-06-01", "2025-06-01"

os.makedirs(file_path, exist_ok=True)   # create the data folder if it doesn't exist

SyntaxError: invalid syntax (1052886889.py, line 5)

## 1. Data pulls

Downloads every dataset from the Elexon API and saves it before any filtering, so it can be checked later. 
This only needs running once; the sections below read the saved csvs. 
The full pull takes a while.

In [ ]:
# BOA: bid-offer acceptances and their flag
r = requests.get(f"{base_url}/datasets/BOALF/stream?from={START}T00:00Z&to={END}T00:00Z")
pd.DataFrame(r.json()).to_csv(f"{file_path}/boalf_2022-06_2025-06.csv", index=False)

# BMU IDs: wind farm identifiers
bmunits = pd.DataFrame(requests.get(f"{base_url}/reference/bmunits/all").json())
bmunits[bmunits['fuelType'] == 'WIND'].to_csv(f"{file_path}/wind_bmunits.csv", index=False)

wind_ids = pd.read_csv(f"{file_path}/wind_bmunits.csv")['elexonBmUnit']
units = "&".join(f"bmUnit={u}" for u in wind_ids)   # url format: &bmUnit=wind_id, repeated for each unit

In [ ]:
# PNs: physical notifications. Pulling 3 years at once returns an empty DataFrame, so pull yearly
years = [("2022-06-01", "2023-06-01"), ("2023-06-01", "2024-06-01"), ("2024-06-01", "2025-06-01")]
pn = pd.DataFrame()
for start, end in years:
    r = requests.get(f"{base_url}/datasets/PN/stream?from={start}&to={end}&{units}")
    print(start, r.status_code, len(r.json()))   # status of pull and number of records returned
    pn = pd.concat([pn, pd.DataFrame(r.json())], ignore_index=True)
pn.drop_duplicates().to_csv(f"{file_path}/pn_wind.csv", index=False)   # drop repeated rows on year boundaries

# Metered generation (B1610). Range limit: a year fails (error 400), so pull monthly
months = pd.date_range(START, END, freq="MS")   # 1st of each month
metered_gen = pd.DataFrame()
for start, end in zip(months[:-1], months[1:]):   # pairs: (1 Jun, 1 Jul), (1 Jul, 1 Aug), ...
    r = requests.get(f"{base_url}/datasets/B1610/stream?settlementDateFrom={start:%Y-%m-%dT00:00Z}&settlementDateTo={end:%Y-%m-%dT00:00Z}&{units}")
    print(start.date(), r.status_code, len(r.json()))
    metered_gen = pd.concat([metered_gen, pd.DataFrame(r.json())], ignore_index=True)
metered_gen.drop_duplicates().to_csv(f"{file_path}/metered_gen_wind.csv", index=False)

In [ ]:
# Wind generation forecast (WINDFOR): whole period in one request
r = requests.get(f"{base_url}/datasets/WINDFOR/stream?publishDateTimeFrom={START}T00:00Z&publishDateTimeTo={END}T00:00Z")
pd.DataFrame(r.json()).to_csv(f"{file_path}/windfor.csv", index=False)

# TSDF and NDF: 3 years returns empty, so pull quarterly. TSDF has 18 boundaries (B1-B17, N): keep N (national)
quarters = pd.date_range(START, END, freq="3MS")
tsdf, ndf = pd.DataFrame(), pd.DataFrame()
for start, end in zip(quarters[:-1], quarters[1:]):
    p = f"publishDateTimeFrom={start:%Y-%m-%dT00:00Z}&publishDateTimeTo={end:%Y-%m-%dT00:00Z}"
    tsdf = pd.concat([tsdf, pd.DataFrame(requests.get(f"{base_url}/datasets/TSDF/stream?{p}&boundary=N").json())], ignore_index=True)
    ndf = pd.concat([ndf, pd.DataFrame(requests.get(f"{base_url}/datasets/NDF/stream?{p}").json())], ignore_index=True)
    print(start.date(), len(tsdf), len(ndf))
tsdf.to_csv(f"{file_path}/tsdf.csv", index=False)
ndf.to_csv(f"{file_path}/ndf.csv", index=False)

# INDO and ITSDO: 28 days per request
chunks = pd.date_range(START, END, freq="28D").append(pd.DatetimeIndex([END]))
indo, itsdo = pd.DataFrame(), pd.DataFrame()
for start, end in zip(chunks[:-1], chunks[1:]):
    p = f"publishDateTimeFrom={start:%Y-%m-%dT00:00Z}&publishDateTimeTo={end:%Y-%m-%dT00:00Z}"
    indo = pd.concat([indo, pd.DataFrame(requests.get(f"{base_url}/datasets/INDO?{p}").json()['data'])], ignore_index=True)
    itsdo = pd.concat([itsdo, pd.DataFrame(requests.get(f"{base_url}/datasets/ITSDO?{p}").json()['data'])], ignore_index=True)
indo.to_csv(f"{file_path}/indo.csv", index=False)
itsdo.to_csv(f"{file_path}/itsdo.csv", index=False)

# Interconnectors: 7 day limit per request (both dates inclusive)
ic = pd.DataFrame()
for start in pd.date_range(START, "2025-05-31", freq="7D"):
    end = min(start + pd.Timedelta(days=6), pd.Timestamp("2025-05-31"))
    r = requests.get(f"{base_url}/generation/outturn/interconnectors?settlementDateFrom={start:%Y-%m-%d}&settlementDateTo={end:%Y-%m-%d}")
    ic = pd.concat([ic, pd.DataFrame(r.json()['data'])], ignore_index=True)
ic.to_csv(f"{file_path}/interconnectors.csv", index=False)

## 2. BOA: wind farm acceptances

| Column | Meaning |
|---|---|
| bmUnit | wind farm unit ID |
| settlementDate | date (UK settlement day) |
| settlementPeriodFrom / To | half-hours the instruction covers, 1-48 |
| timeFrom / timeTo | start and end of the instruction (UTC) |
| levelFrom / levelTo | level NESO instructed (MW) |
| acceptanceNumber | ID of the instruction |
| acceptanceTime | when NESO issued the instruction |
| soFlag | True = constraint action |

In [ ]:
boa = pd.read_csv(f"{file_path}/boalf_2022-06_2025-06.csv")

# keep only the columns needed to build curtailment
cols = ['bmUnit', 'settlementDate', 'settlementPeriodFrom', 'settlementPeriodTo',
        'timeFrom', 'timeTo', 'levelFrom', 'levelTo',
        'acceptanceNumber', 'acceptanceTime', 'soFlag']
boa = boa[cols]

# convert text times to datetimes (the Z means UTC)
for c in ['timeFrom', 'timeTo', 'acceptanceTime']:
    boa[c] = pd.to_datetime(boa[c], utc=True)
boa['settlementDate'] = pd.to_datetime(boa['settlementDate'])

# filter to wind farms only
wind_ids = pd.read_csv(f"{file_path}/wind_bmunits.csv")['elexonBmUnit']
boa_wind = boa[boa['bmUnit'].isin(wind_ids)]

print("all acceptances:", boa.shape)
print("wind acceptances:", boa_wind.shape)
print("wind units registered:", wind_ids.nunique())
print("wind units with at least one acceptance:", boa_wind['bmUnit'].nunique())

## 3. PN: physical notifications

| Column | Meaning |
|---|---|
| bmUnit | wind farm unit ID |
| settlementDate | date (UK settlement day) |
| settlementPeriod | half-hour the plan covers, 1-48 |
| timeFrom / timeTo | start and end of the half-hour (UTC) |
| levelFrom / levelTo | level the unit planned (MW) |

In [ ]:
pn = pd.read_csv(f"{file_path}/pn_wind.csv")

# remove unnecessary columns, convert time columns to datetime format as done with boa dataset
pn = pn.drop(columns=['dataset', 'nationalGridBmUnit'])
for c in ['timeFrom', 'timeTo']:
    pn[c] = pd.to_datetime(pn[c], utc=True)
pn['settlementDate'] = pd.to_datetime(pn['settlementDate'])

print(pn.shape)

## 4. Calculate curtailment

Curtailment = (PN − accepted level) × time, where PN is higher (Storey et al.).

NESO can change accepted levels within a time interval. The latest one replaces the previous, so where a farm's acceptances overlap, the earlier one is cut where the next one starts.

In [ ]:
# all of each wind farm's acceptances, in time order (acceptanceNumber breaks ties: later acceptance second)
boa_sorted = boa_wind.sort_values(['bmUnit', 'timeFrom', 'acceptanceNumber']).copy()

# start time of the next acceptance for the same farm
next_start = boa_sorted.groupby('bmUnit')['timeFrom'].shift(-1)

# overlap: next acceptance starts before this one ends
cut = next_start < boa_sorted['timeTo']
print("overlapping acceptances:", cut.sum())

# how far along the old acceptance had gone before the new acceptance interrupted
fraction = (next_start - boa_sorted['timeFrom']) / (boa_sorted['timeTo'] - boa_sorted['timeFrom'])

# level at the cut (calculated levels have decimals, so change level cols from integer to float)
boa_sorted[['levelFrom', 'levelTo']] = boa_sorted[['levelFrom', 'levelTo']].astype(float)
boa_sorted.loc[cut, 'levelTo'] = boa_sorted['levelFrom'] + (boa_sorted['levelTo'] - boa_sorted['levelFrom']) * fraction

# end the earlier acceptance where the next one starts
boa_sorted.loc[cut, 'timeTo'] = next_start

In [ ]:
# 1. how long each instruction lasted, and its average level
boa_sorted['hours'] = (boa_sorted['timeTo'] - boa_sorted['timeFrom']).dt.total_seconds() / 3600
boa_sorted['boaLevel'] = (boa_sorted['levelFrom'] + boa_sorted['levelTo']) / 2

# 2. planned level (PN) for each farm and half-hour
pn['pnLevel'] = (pn['levelFrom'] + pn['levelTo']) / 2
pn_sp = pn.groupby(['bmUnit', 'settlementDate', 'settlementPeriod'], as_index=False)['pnLevel'].mean()

# merge PN level onto accepted levels (inner: keeps acceptances that have a matching PN)
curtailment = boa_sorted.merge(pn_sp, left_on=['bmUnit', 'settlementDate', 'settlementPeriodFrom'],
                               right_on=['bmUnit', 'settlementDate', 'settlementPeriod'], how='inner')

# 3. curtailed MWh = (planned - instructed) x hours, only where planned is higher --> clip lower=0
curtailment['curtailed_MWh'] = (curtailment['pnLevel'] - curtailment['boaLevel']).clip(lower=0) * curtailment['hours']

# 4. hour the instruction began in (UTC)
curtailment['hour'] = curtailment['timeFrom'].dt.floor('h')

n_rows = len(curtailment)   # every feature merge below should keep this number of rows
print("acceptances with a matching PN:", n_rows, "of", len(boa_sorted))
print("wind units after PN merge:", curtailment['bmUnit'].nunique())
print("acceptances with curtailment > 0 (SO / non-SO):")
print(curtailment[curtailment['curtailed_MWh'] > 0]['soFlag'].value_counts())

## 5. Features

Day-ahead framing: forecasts must be published before the target day; actual values (metered generation, INDO, ITSDO, interconnectors) are only known after the fact, so they are lagged one day.

### Metered generation (B1610), per wind farm, lagged one day

In [ ]:
metered_gen = pd.read_csv(f"{file_path}/metered_gen_wind.csv")
metered_gen = metered_gen[['bmUnit', 'settlementDate', 'settlementPeriod', 'settlementRunType', 'quantity']]
metered_gen['settlementDate'] = pd.to_datetime(metered_gen['settlementDate'])

# data is refreshed by later settlement runs: keep the latest run for each unit and half-hour
print(metered_gen['settlementRunType'].unique())   # check all runs are in run_order
run_order = {'II': 0, 'SF': 1, 'R1': 2, 'R2': 3, 'R3': 4, 'RF': 5, 'DF': 6}
metered_gen['run'] = metered_gen['settlementRunType'].map(run_order)
metered_gen = metered_gen.sort_values('run').drop_duplicates(['bmUnit', 'settlementDate', 'settlementPeriod'], keep='last')

# lag one day: each row gets the previous day's metered generation for the same half-hour
metered_gen['settlementDate'] = metered_gen['settlementDate'] + pd.Timedelta(days=1)
metered_gen = metered_gen[['bmUnit', 'settlementDate', 'settlementPeriod', 'quantity']].rename(columns={'quantity': 'metered_MWh_lag1d'})

curtailment = curtailment.merge(metered_gen, on=['bmUnit', 'settlementDate', 'settlementPeriod'], how='left')
print(len(curtailment) == n_rows, curtailment['metered_MWh_lag1d'].isna().sum())

### Wind generation forecast (WINDFOR): GB total, same value for every wind farm in the same hour

In [ ]:
wind_forecast = pd.read_csv(f"{file_path}/windfor.csv")
wind_forecast['publishTime'] = pd.to_datetime(wind_forecast['publishTime'], utc=True)
wind_forecast['startTime'] = pd.to_datetime(wind_forecast['startTime'], utc=True)

# day-ahead: keep the latest forecast published at least 24 hours before each startTime
wind_forecast = wind_forecast[wind_forecast['startTime'] - wind_forecast['publishTime'] >= pd.Timedelta(hours=24)]
wind_forecast = wind_forecast.sort_values('publishTime').drop_duplicates('startTime', keep='last')

# forecast is already hourly: rename to match curtailment
wind_forecast = wind_forecast.rename(columns={'startTime': 'hour', 'generation': 'wind_forecast_MW'})

curtailment = curtailment.merge(wind_forecast[['hour', 'wind_forecast_MW']], on='hour', how='left')
print(len(curtailment) == n_rows, curtailment['wind_forecast_MW'].isna().sum())

### TSDF and NDF: day-ahead demand forecasts

In [ ]:
keys = ['settlementDate', 'settlementPeriod']

for file, name in [("tsdf.csv", 'tsdf_MW'), ("ndf.csv", 'ndf_MW')]:
    df = pd.read_csv(f"{file_path}/{file}")
    df['publishTime'] = pd.to_datetime(df['publishTime'], utc=True)
    df['startTime'] = pd.to_datetime(df['startTime'], utc=True)

    # day-ahead: latest forecast published before the target day starts
    # e.g. a forecast for 15:30 on 3 June is kept if published on 2 June or earlier, dropped if published on 3 June
    df = df[df['publishTime'] < df['startTime'].dt.floor('D')]
    df = df.sort_values('publishTime').drop_duplicates('startTime', keep='last')

    df['settlementDate'] = pd.to_datetime(df['settlementDate'])
    curtailment = curtailment.merge(df[keys + ['demand']].rename(columns={'demand': name}), on=keys, how='left')

print(len(curtailment) == n_rows, curtailment[['tsdf_MW', 'ndf_MW']].isna().sum().tolist())

### INDO, ITSDO and interconnectors: actual values, lagged one day

Missing lagged values fall on clock-change days: October days have 50 periods (49 and 50 have nothing to lag from), and the day after the March change follows a 46-period day.

In [ ]:
def lag_1day(df, value, name):
    df = df.copy()
    df['settlementDate'] = pd.to_datetime(df['settlementDate']) + pd.Timedelta(days=1)
    return df[keys + [value]].rename(columns={value: name})

indo = pd.read_csv(f"{file_path}/indo.csv").drop_duplicates(keys)
itsdo = pd.read_csv(f"{file_path}/itsdo.csv").drop_duplicates(keys)

# interconnectors: total flow across all interconnectors per settlement period
ic = pd.read_csv(f"{file_path}/interconnectors.csv")
ic_total = ic.groupby(keys, as_index=False)['generation'].sum()

for lagged in [lag_1day(indo, 'demand', 'indo_MW_lag1d'),
               lag_1day(itsdo, 'demand', 'itsdo_MW_lag1d'),
               lag_1day(ic_total, 'generation', 'interconnector_MW_lag1d')]:
    curtailment = curtailment.merge(lagged, on=keys, how='left')

lag_cols = ['indo_MW_lag1d', 'itsdo_MW_lag1d', 'interconnector_MW_lag1d']
print(len(curtailment) == n_rows, curtailment[lag_cols].isna().sum().tolist())
print(curtailment.loc[curtailment['indo_MW_lag1d'].isna(), 'settlementDate'].value_counts())

## 6. Save

| Column | Meaning |
|---|---|
| bmUnit, settlementDate, settlementPeriodFrom/To, timeFrom/timeTo, acceptanceNumber, acceptanceTime, soFlag | same as BOA |
| levelFrom / levelTo | level NESO instructed (MW), with earlier instructions cut where a later one starts |
| hours | how long the instruction lasted |
| boaLevel | average instructed level over the instruction (MW) |
| settlementPeriod | the PN half-hour matched to the instruction |
| pnLevel | average planned level for that half-hour (MW) |
| curtailed_MWh | (pnLevel − boaLevel) × hours, 0 where planned is not higher |
| hour | start of the hour the instruction began in (UTC) |
| metered_MWh_lag1d | the farm's metered generation, same half-hour previous day |
| wind_forecast_MW | GB wind forecast for that hour, published ≥24h before |
| tsdf_MW, ndf_MW | day-ahead demand forecasts for that half-hour |
| indo_MW_lag1d, itsdo_MW_lag1d | actual demand, same half-hour previous day |
| interconnector_MW_lag1d | total interconnector flow, same half-hour previous day |

In [ ]:
print(curtailment.shape)
curtailment.to_csv(f"{file_path}/elexon_data.csv", index=False)
curtailment.head()